# Probability and Statistics Term Project
## World Happiness Report 2015 Dataset

This notebook applies core Probability and Statistics concepts to the **2015 World Happiness Report** dataset.

## Dataset Source
World Happiness Report 2015 (local CSV file)

## Main Variable of Interest
- `Happiness Score` — the overall happiness index per country

## Important Predictor
- `Economy (GDP per Capita)` — strongest numerical predictor (r = 0.78)

## Topics Covered
1. Data loading and preprocessing
2. Descriptive statistics
3. Histogram and boxplot
4. Probability Density Function (PDF)
5. Empirical Cumulative Distribution Function (CDF)
6. Probability calculations
7. Conditional probability
8. Sampling
9. Confidence intervals
10. Hypothesis testing
11. Correlation
12. Linear regression

In [ ]:
# Cell 1: Import Required Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## Step 1: Load and Prepare the Dataset

In [ ]:
# Cell 2: Load and Prepare the Dataset

# Load the 2015 World Happiness Report CSV
df = pd.read_csv('2015.csv')  # Place the CSV file in the same folder as this notebook

# Keep only numerical columns (drop Country and Region text columns)
df_num = df.select_dtypes(include=[np.number])

# Fill any missing values with column medians (safe default)
df_num = df_num.fillna(df_num.median())

print('Dataset shape:', df_num.shape)
display(df_num.head())

## Step 2: Define Main Variable (Happiness Score)

In [ ]:
# Cell 3: Define Main Variable

x = df_num['Happiness Score']

print('First 5 values:')
print(x.head())

## Step 3: Descriptive Statistics

In [ ]:
# Cell 4: Descriptive Statistics

print('Mean    :', round(x.mean(), 4))
print('Median  :', round(x.median(), 4))
print('Mode    :', round(x.mode()[0], 4))
print('Variance:', round(x.var(), 4))
print('Std Dev :', round(x.std(), 4))
print('Min     :', x.min())
print('Max     :', x.max())

print('\nFull Summary Statistics:')
display(df_num.describe())

## Step 4: Outlier Detection Using IQR

In [ ]:
# Cell 5: Identify Outliers Using the IQR Method

Q1 = x.quantile(0.25)
Q3 = x.quantile(0.75)
IQR = Q3 - Q1

Lower_Bound = Q1 - 1.5 * IQR
Upper_Bound = Q3 + 1.5 * IQR

outliers = x[(x < Lower_Bound) | (x > Upper_Bound)]
x_clean = x[(x >= Lower_Bound) & (x <= Upper_Bound)]

print('Outlier Detection Using the IQR Method')
print('-' * 45)
print(f'Q1 = {Q1:.4f},  Q3 = {Q3:.4f},  IQR = {IQR:.4f}')
print(f'Lower Bound = {Lower_Bound:.4f},  Upper Bound = {Upper_Bound:.4f}')
print(f'Number of Outliers = {len(outliers)} ({len(outliers)/len(x)*100:.2f}%)')

# Boxplots: Before vs After
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.boxplot(x, vert=False)
plt.title('Before Outlier Removal')
plt.xlabel('Happiness Score')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(x_clean, vert=False, showfliers=False)
plt.title('After Outlier Removal (IQR)')
plt.xlabel('Happiness Score')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

if len(outliers) > 0:
    print('\nOutlier Values:')
    print(outliers.values)
else:
    print('\nNo outliers detected — the data is well-distributed.')

## Step 5: Histogram with Normal PDF Overlay

In [ ]:
# Cell 6: Histogram with Normal PDF Overlay

mu = x.mean()
sigma = x.std()

grid = np.linspace(x.min(), x.max(), 300)
pdf = stats.norm.pdf(grid, mu, sigma)

plt.figure(figsize=(8, 5))
plt.hist(x, bins=20, density=True, edgecolor='black', alpha=0.6, label='Happiness Score Data')
plt.plot(grid, pdf, color='red', linewidth=3, label=f'Normal PDF (μ={mu:.2f}, σ={sigma:.2f})')
plt.xlabel('Happiness Score')
plt.ylabel('Probability Density')
plt.title('Histogram of Happiness Scores with Normal PDF Overlay')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Skewness analysis
skew = x.skew()
print(f'Skewness = {skew:.4f}')
if abs(skew) < 0.5:
    print('Distribution is approximately symmetric (close to normal).')
elif skew > 0:
    print('Distribution is right-skewed (positive skew): tail extends to higher scores.')
else:
    print('Distribution is left-skewed (negative skew): tail extends to lower scores.')

## Step 6: Empirical Cumulative Distribution Function (CDF)

In [ ]:
# Cell 7: Empirical CDF

x_sorted = np.sort(x)
cdf = np.arange(1, len(x_sorted) + 1) / len(x_sorted)

plt.figure(figsize=(8, 5))
plt.plot(x_sorted, cdf, color='steelblue', linewidth=2)
plt.xlabel('Happiness Score')
plt.ylabel('Cumulative Probability')
plt.title('Empirical CDF of Happiness Scores')
plt.grid(alpha=0.3)
plt.show()

## Step 7: Empirical Probability

**Target threshold**: Happiness Score > 6.0  
A score above 6.0 is considered **above average happiness** — roughly the top 30% globally.

In [ ]:
# Cell 8: Empirical Probability Calculation

threshold = 6.0
prob = np.mean(x > threshold)

print(f'P(Happiness Score > {threshold}) = {prob:.4f}')
print(f'Percentage of countries with Happiness Score > {threshold}: {prob*100:.2f}%')

## Step 8: Conditional Probability + 2D Joint Distribution

**Condition (A):** Economy (GDP per Capita) > 1.0  
**Event (B):** Happiness Score > 6.0  

**Question:** Given a country has a high GDP contribution (> 1.0), what is the probability its Happiness Score also exceeds 6.0?

In [ ]:
# Cell 9: Conditional Probability
#
# Formula:
# P(B | A) = Number(A and B) / Number(A)
#
# A: Economy (GDP per Capita) > 1.0
# B: Happiness Score > 6.0

A = df_num['Economy (GDP per Capita)'] > 1.0
B = df_num['Happiness Score'] > 6.0

n_A       = np.sum(A)
n_A_and_B = np.sum(A & B)

P_B_given_A = n_A_and_B / n_A

print('P(B | A) = Number(A and B) / Number(A)')
print(f'Number of countries with GDP > 1.0              = {n_A}')
print(f'Number of countries with GDP > 1.0 AND Score > 6.0 = {n_A_and_B}')
print(f'P(Happiness > 6.0 | GDP > 1.0) = {P_B_given_A:.4f}')
print(f'Percentage = {P_B_given_A * 100:.2f}%')

# 2D Joint Distribution Histogram
plt.figure(figsize=(8, 6))
plt.hist2d(df_num['Economy (GDP per Capita)'], df_num['Happiness Score'], bins=20, cmap='Blues')
plt.axvline(x=1.0, color='red', linestyle='--', linewidth=1.5, label='GDP Threshold = 1.0')
plt.axhline(y=6.0, color='orange', linestyle='--', linewidth=1.5, label='Score Threshold = 6.0')
plt.xlabel('Economy (GDP per Capita)')
plt.ylabel('Happiness Score')
plt.title('2D Joint Distribution: GDP vs Happiness Score')
plt.colorbar(label='Frequency')
plt.legend()
plt.show()

## Step 9: Sampling Distribution (Central Limit Theorem)

In [ ]:
# Cell 10: Sampling Distribution of the Mean

sample_means = []

for i in range(1000):
    sample = x.sample(n=50, replace=True)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

plt.figure(figsize=(8, 5))
plt.hist(sample_means, bins=30, density=True, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(x.mean(), color='red', linestyle='--', linewidth=2, label=f'Population Mean = {x.mean():.4f}')
plt.xlabel('Sample Mean of Happiness Score')
plt.ylabel('Density')
plt.title('Sampling Distribution of the Mean (n=50, 1000 samples)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f'Population Mean        : {x.mean():.4f}')
print(f'Mean of Sample Means   : {sample_means.mean():.4f}')
print(f'Std of Sample Means    : {sample_means.std():.4f}')

## Step 10: 95% Confidence Interval for the Population Mean

**Formula:** CI = x̄ ± z × (s / √n)

In [ ]:
# Cell 11: 95% Confidence Interval (n=100)

sample = x.sample(n=100, random_state=42)

n     = len(sample)
x_bar = sample.mean()
s     = sample.std(ddof=1)
SE    = s / np.sqrt(n)

confidence = 0.95
alpha = 1 - confidence
z = stats.norm.ppf(1 - alpha / 2)
ME = z * SE

lower = x_bar - ME
upper = x_bar + ME

print('95% Confidence Interval Formula:')
print('CI = x̄ ± z × (s / √n)\n')
print(f'Confidence Level      = {confidence*100:.0f}%')
print(f'Alpha (α)             = {alpha:.2f}')
print(f'Critical Value (z)    = {z:.4f}\n')
print(f'Sample Mean (x̄)      = {x_bar:.4f}')
print(f'Sample Std Dev (s)    = {s:.4f}')
print(f'Sample Size (n)       = {n}')
print(f'Standard Error (SE)   = {SE:.4f}')
print(f'Margin of Error (ME)  = {ME:.4f}\n')
print(f'95% Confidence Interval = ({lower:.4f}, {upper:.4f})')

## Step 11: Hypothesis Testing (Two-Tailed Z-Test)

**H₀ (Null Hypothesis):** The mean global happiness score = 5.5 (assumed historical baseline)  
**H₁ (Alternative Hypothesis):** The mean global happiness score ≠ 5.5  
**Significance level:** α = 0.05  
**Test type:** Two-tailed Z-test

In [ ]:
# Cell 12: Two-Tailed Z-Test

# H0: Mean Happiness Score = 5.5
# H1: Mean Happiness Score != 5.5
null_mean = 5.5

sample_mean = x.mean()
n           = len(x)
sample_std  = x.std()
SE          = sample_std / np.sqrt(n)

z_stat  = (sample_mean - null_mean) / SE
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print('--- Z-Test Results ---')
print(f'H0: μ = {null_mean}')
print(f'H1: μ ≠ {null_mean}\n')
print(f'Sample Mean (x̄)   : {sample_mean:.4f}')
print(f'Standard Error (SE): {SE:.4f}')
print(f'Z-statistic        : {z_stat:.4f}')
print(f'p-value            : {p_value:.4f}\n')

if p_value < 0.05:
    print('Decision: Reject H0')
    print(f'The mean happiness score is significantly different from {null_mean} (p < 0.05).')
else:
    print('Decision: Fail to Reject H0')
    print(f'There is not enough evidence to say the mean differs from {null_mean}.')

## Step 12: Correlation Matrix

In [ ]:
# Cell 13: Full Correlation Matrix

corr_matrix = df_num.corr()

# Display correlations with Happiness Score sorted
print('Correlation with Happiness Score (sorted):')
display(corr_matrix['Happiness Score'].sort_values(ascending=False).to_frame())

# Heatmap
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_matrix.columns, fontsize=9)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix.columns)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.title('Correlation Matrix Heatmap')
plt.tight_layout()
plt.show()

## Step 13: Simple Linear Regression

**Predictor (X):** Economy (GDP per Capita)  
**Target (y):** Happiness Score

In [ ]:
# Cell 14: Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df_num[['Economy (GDP per Capita)']]
y = df_num['Happiness Score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f'Regression Equation: Happiness Score = {model.coef_[0]:.4f} × GDP + {model.intercept_:.4f}')
print(f'Slope (β₁)    : {model.coef_[0]:.4f}')
print(f'Intercept (β₀): {model.intercept_:.4f}')
print(f'R² Score      : {r2_score(y_test, y_pred):.4f}')

## Step 14: Actual vs Predicted Happiness Score Plot

In [ ]:
# Cell 15: Actual vs Predicted Plot

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white')

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Actual Happiness Score')
plt.ylabel('Predicted Happiness Score')
plt.title('Actual vs Predicted Happiness Score')
plt.legend()
plt.grid(alpha=0.3)
plt.show()